# More care. Better health?

**A guided investigation · Chapter 1 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-rand_health_outcomes.ipynb)

> **The question**  
> How do measured health outcomes compare across insurance plans?

Using more healthcare and becoming healthier are different outcomes. The RAND Health Insurance Experiment makes that distinction tangible: families were assigned different cost-sharing plans, and researchers collected health measures as well as spending records. This extract contains exit measures such as a general health index, blood pressure, and cholesterol. A row represents a participant in the stored exit-measure file, but some rows lack the measurements needed for a particular comparison. Different outcomes also have different scales and interpretations; higher values are not always better. We will first make the available evidence visible, then compare the distributions of selected health measures across plans. The aim is to learn how to ask a precise outcome question and avoid turning one descriptive chart into a broad claim about every participant or every dimension of health.

### By the end, you will be able to…

- Compare health outcomes and check whether missing measurements limit the comparison.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

If a plan produces more medical visits, must it improve every health measure?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** RAND HIE participant with exit health measures.  
**Stored size:** 5,887 rows × 10 columns.

| Column | What it records |
|---|---|
| `plan_type` | Assigned plan: 1 free care, 2 deductible, 3 coinsurance, 4 catastrophic coverage. Missing means unknown. |
| `health_index` | Source general health index; higher values indicate better reported health. Distinct from the NHIS 1–5 score. |
| `blood_pressure` | Source systolic blood pressure measurement. Clinical unit is not independently documented in this CSV. |
| `cholesterol` | Source blood cholesterol measurement. Clinical unit is not independently documented in this CSV; do not infer it from the numeric scale. |

Outcome availability differs across measures. Do not combine health indices, blood pressure, and cholesterol on one numeric scale. These plots are not clinical advice or estimates of statistical significance.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch1/rand_health_outcomes.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['plan_type', 'health_index', 'blood_pressure', 'cholesterol']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['plan_type', 'health_index', 'blood_pressure', 'cholesterol']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability_group': 'Ability group',
 'blood_pressure': 'Systolic blood pressure (source units)',
 'census': 'Census year',
 'cholesterol': 'Cholesterol (source units)',
 'group': 'State group',
 'health_index': 'General health index (source scale)',
 'offer': 'Scholarship assignment',
 'plan': 'Assigned insurance plan',
 'state_name': 'Simulated state'}

### Keep known plan assignments

We exclude unknown plan types from plan comparisons and translate codes into readable labels. `notna()` identifies nonmissing assignments; `map` replaces a code with a label. The earlier missing-value check still includes the full stored file.

In [ ]:
plans = df[df["plan_type"].notna()].copy()
plans["plan"] = plans["plan_type"].map({1: "Free care", 2: "Deductible", 3: "Coinsurance", 4: "Catastrophic"})

## Visual investigation 1 · First, count the available evidence

**Predict:** How many usable records support each plan comparison?

We keep rows with a known plan and count available values for this outcome. count excludes missing measurements. A small bar could mean fewer participants or more missing outcomes, so it is useful to check both.

In [ ]:
coverage = plans.groupby("plan", as_index=False).agg(records=("plan", "size"), observed=("health_index", "count"))
coverage["missing"] = coverage["records"] - coverage["observed"]
coverage_long = coverage.melt(id_vars="plan", value_vars=["observed", "missing"], var_name="availability", value_name="participants")
fig = px.bar(coverage_long, x="plan", y="participants", color="availability", labels={"plan": "Assigned plan", "participants": "Participant records", "availability": "health_index availability"}, title="What evidence is available?")
fig.show()

**Explore the chart:** Hover over the observed and missing segments. Missing measurements are not zero-valued outcomes.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Keep the availability chart beside your substantive comparison. Different sample sizes and missingness can affect how confidently we compare groups.

</details>

## Visual investigation 2 · Look beyond a single average

**Predict:** How does health index vary within and across plans?

A box contains the middle half of the available values; its central line is the median. The whiskers extend into the tails. We suppress individual points to make the groups readable.

In [ ]:
fig = px.box(plans, x="plan", y="health_index", color="plan", points=False, labels=labels, title="Health at the end of the experiment")
fig.show()

**Explore the chart:** Hover over the boxes. Compare the differences between medians with the spread within each plan.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Overlapping distributions remind us that people vary within every plan. A descriptive difference is not a significance test and is not automatically evidence of a large effect.

</details>

## Visual investigation 3 · Ask whether the picture travels

**Predict:** Does the comparison look similar for blood pressure?

groupby gathers the known-plan records; mean averages available values within each plan. The source rows used can differ from the previous chart because this outcome has its own missing values.

In [ ]:
means = plans.groupby("plan", as_index=False)["blood_pressure"].mean()
fig = px.bar(means, x="plan", y="blood_pressure", color="plan", labels=labels, title="A second measure, a separate comparison")
fig.show()

**Explore the chart:** Hover for exact means. Use the variable guide to decide what a higher value means for this measure.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Do not generalize from one outcome to all outcomes. Ask whether the records are comparable, whether the measure is relevant, and what the research design can justify.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

Change the box plot outcome from health_index to cholesterol. Read its definition before interpreting it.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Calculate and plot the number of nonmissing cholesterol records per plan.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Use the plans table, which excludes unknown plan types.

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

groupby(...).count() counts nonmissing observations for your chosen variable.

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>counts = plans.groupby(&quot;plan&quot;, as_index=False)[&quot;cholesterol&quot;].count()
fig = px.bar(counts, x=&quot;plan&quot;, y=&quot;cholesterol&quot;, labels={&quot;plan&quot;: &quot;Assigned plan&quot;, &quot;cholesterol&quot;: &quot;Available participant records&quot;}, title=&quot;Check the denominator&quot;)
fig.show()</code></pre>

Counts tell you how much available evidence lies behind a comparison. They do not tell you whether missingness is unrelated to treatment or health.

Copy the code into the starter cell to run it.

</details>

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**How could missing exit measurements weaken a comparison that began with random assignment?**

[Continue with Chapter 1: Randomized Trials](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/01-randomized-trials.html#step-3-estimate-causal-effects-on-health) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch1-rand-health-outcomes-csv)

### Source and preparation notes

[Aron-Dine, Einav and Finkelstein (2013), The RAND Health Insurance Experiment, Three Decades Later](https://www.aeaweb.org/articles?id=10.1257/jep.27.1.197)

The preparation script maps exit health measures, plan indicators and numeric family ID from the RAND source. Free-plan indicator is set missing when plan type is missing. Profiles are unweighted.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.